# Assignment 1: False Alarms

Author: Fredrik Boglind

## Introduction

Alarm fatigue is a critical problem in intensive care units (ICUs), where clinical staff are exposed to frequent monitor alarms — many of which are false positives. This assignment uses the **VTaC dataset** (Ventricular Tachycardia annotated alarms from ICUs) from [PhysioNet](https://physionet.org/content/vtac/) to build a classifier that predicts whether a ventricular tachycardia (VT) alarm is **True** or **False**.

Each alarm event in the dataset contains up to four physiological waveforms sampled at 250 Hz:
- **ECG Lead 1** and **ECG Lead 2** — electrocardiogram signals
- **PLETH (PPG)** — photoplethysmogram
- **ABP** — arterial blood pressure

The dataset provides a benchmark train/validation/test split and expert-annotated alarm labels.

### Imports

In [1]:
import pandas as pd
import numpy as np
import wfdb
import os
from pathlib import Path
from scipy.signal import butter, filtfilt, iirnotch

import sklearn as sk
from sklearn.model_selection import cross_val_score
import seaborn as sns
import matplotlib.pyplot as plt


# %matplotlib inline
# plt.rcParams['figure.figsize'] = (12, 4)
print("All imports successful.")

All imports successful.


## Data Loading

Beside the waveform data we have two metadata csv files to work with:

`event_labels.csv` has three columns:
- `record`:
- `event`:
- `decision`: 

    a list that says "alarm #003c13_0115 was a false alarm" and "alarm #007ab2_0042 was a real alarm" — one row per alarm event, with a True/False answer.

- `benchmark_data_split.csv` also contains the `record` and `event_columns` and has a column `split` that assigns each event to a train/val/test-split. 


In [ ]:
DATA_DIR = Path('raw_data/vtac-a-benchmark-dataset-of-ventricular-tachycardia-alarms-from-icu-monitors-1.0/')

# Load metadata
labels_df = pd.read_csv(DATA_DIR / 'event_labels.csv')
splits_df = pd.read_csv(DATA_DIR / 'benchmark_data_split.csv')

In [18]:
labels_df.head()
labels_df.shape

(5037, 3)

In [20]:
splits_df.head()
#splits_df.shape

,split,record,event
0,train,003c13,003c13_0115
1,train,003c13,003c13_0126
2,train,004bad,004bad_0015
3,train,004bad,004bad_1115
4,train,004bad,004bad_1426


We create a unified dataframe by merging the the files on `record` and `event`. Then we check the shape of the dataframes,

In [32]:
# Merge to get labels with split assignments
df = splits_df.merge(labels_df, on=['record', 'event'], how='inner')

# Shape of original dfs
print(f"Labels:  {labels_df.shape}")
print(f"Splits:  {splits_df.shape}")
# Shape of merged df
print(f"Merged:  {df.shape}")
print(f"\nColumns after merge:\n{df.columns.tolist()}")
df.head()

Labels:  (5037, 3)
Splits:  (5037, 3)
Merged:  (5037, 4)

Columns after merge:
['split', 'record', 'event', 'decision']


,split,record,event,decision
0,train,003c13,003c13_0115,False
1,train,003c13,003c13_0126,False
2,train,004bad,004bad_0015,False
3,train,004bad,004bad_1115,False
4,train,004bad,004bad_1426,False


## Task 1: Predict whether or not an alarm is True or False

### Data Exploration

Before we load the waveforrms and start modelling we must understand the structure, size, and balance of the datasets.

#### Dataset overview and class balance

In [ ]:
print(f"Total alarm events: {len(df)}")
print(f"Unique records: {df['record'].nunique()}")
print(f"Events per record: {df.groupby('record')['event'].count().describe().to_string()}")

print(f"\nClass distribution")
print(df['decision'].value_counts())
true_pct = df['decision'].mean() * 100
print(f"\nClass balance: {true_pct:.1f}% True alarms, {100-true_pct:.1f}% False alarms")

Total alarm events: 5037
Unique records (patients/sessions): 2260
Events per record: count    2260.000000
mean        2.228761
std         1.606308
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max        21.000000

--- Class distribution ---
decision
False    3596
True     1441
Name: count, dtype: int64

Class balance: 28.6% True alarms, 71.4% False alarms


The dataset is imbalanced, 71% of alarms are false and 29% are true. This reflects the real clinical setting where most VT alarms are false positives. We need to account for this imbalance when training and evaluating models (some ways to do this would be: using stratified splits, class weights, and appropriate metrics, i e F1-score).

#### Split distribution

In [ ]:
# Cross-tab of splits and labels
ct = pd.crosstab(df['split'], df['decision'], margins=True)
print(ct)

# Check that class proportions are preserved
print("\nClass proportions per split:")
for split in ['train', 'val', 'test']:
    subset = df[df['split'] == split]
    print(f"  {split}: {subset['decision'].mean():.1%} True  (n={len(subset)})")

decision  False  True   All
split                      
test        345   137   482
train      2897  1163  4060
val         354   141   495
All        3596  1441  5037

Class proportions per split:
  train: 28.6% True  (n=4060)
  val: 28.5% True  (n=495)
  test: 28.4% True  (n=482)


The benchmark split maintains approximately the same class proportions across train, validation, and test sets (~28-29% True), which is important for unbiased evaluation.

#### Waveform signal availability

For each event, there is a corresponding waveform file. It is worth noting that not all events have all four signals (ECG1, ECG2, PLETH, ABP), some channels are entirely missing (zero-values or absent). We scan the waveform files to check what exactly is missing.

In [ ]:
def inspect_waveform_availability(df, data_dir, max_events=None):
    """
    Scan waveform files to determine which signals are available per event.
    
    Returns a DataFrame with boolean columns for each signal type.
    """
    records = []
    events_to_check = df if max_events is None else df.head(max_events)
    
    for _, row in events_to_check.iterrows():
        event = row['event']
        record = row['record']
        wfdb_path = str(data_dir / record / event)
        
        try:
            header = wfdb.rdheader(wfdb_path)
            sig_names = [s.upper() for s in header.sig_name]
            records.append({
                'event': event,
                'n_signals': len(sig_names),
                'signal_names': header.sig_name,
                'has_ecg1': any('II' in s or 'ECG' in s or 'V' == s for s in sig_names),
                'has_ecg2': len([s for s in sig_names if 'II' in s or 'ECG' in s or 'V' == s or 'I' == s or 'III' in s or 'AVR' in s or 'AVL' in s or 'AVF' in s]) >= 2,
                'has_pleth': 'PLETH' in sig_names,
                'has_abp': 'ABP' in sig_names,
                'sig_length': header.sig_len,
            })
        except Exception as e:
            records.append({
                'event': event,
                'n_signals': 0,
                'signal_names': [],
                'has_ecg1': False,
                'has_ecg2': False,
                'has_pleth': False,
                'has_abp': False,
                'sig_length': 0,
            })
    
    return pd.DataFrame(records)

# Scan all events (this may take a moment)
print("Scanning waveform headers...")
avail_df = inspect_waveform_availability(df, DATA_DIR)
print(f"Scanned {len(avail_df)} events.")

In [ ]:
# Signal availability summary
print("=== Signal Availability ===")
for col in ['has_ecg1', 'has_ecg2', 'has_pleth', 'has_abp']:
    signal_name = col.replace('has_', '').upper()
    count = avail_df[col].sum()
    print(f"  {signal_name}: {count}/{len(avail_df)} events ({count/len(avail_df):.1%})")

print(f"\n=== Signal count distribution ===")
print(avail_df['n_signals'].value_counts().sort_index())

print(f"\n=== Waveform length (samples) ===")
print(avail_df['sig_length'].describe())
print(f"Duration at 250 Hz: {avail_df['sig_length'].median() / 250:.0f} seconds (median)")

In [ ]:
# Which signal combinations exist?
avail_df['combo'] = avail_df.apply(
    lambda r: '+'.join([s for s, present in [
        ('ECG1', r['has_ecg1']), ('ECG2', r['has_ecg2']),
        ('PLETH', r['has_pleth']), ('ABP', r['has_abp'])
    ] if present]), axis=1)

print("=== Signal combinations ===")
print(avail_df['combo'].value_counts().to_string())

This tells us which signals are most commonly available and which are frequently missing. ABP tends to be the most commonly absent signal. We will need a strategy for handling events where not all signals are present (see the imputation section below).

#### Visualise example waveforms

In [ ]:
def plot_example_event(event_id, record_id, data_dir, title_suffix=""):
    """Plot all available signals for a single event."""
    wfdb_path = str(data_dir / record_id / event_id)
    record = wfdb.rdrecord(wfdb_path)
    
    sig_names = record.sig_name
    n_sigs = len(sig_names)
    fs = record.fs
    t = np.arange(record.sig_len) / fs
    
    fig, axes = plt.subplots(n_sigs, 1, figsize=(14, 2.5 * n_sigs), sharex=True)
    if n_sigs == 1:
        axes = [axes]
    
    for i, (ax, name) in enumerate(zip(axes, sig_names)):
        signal = record.p_signal[:, i]
        ax.plot(t, signal, linewidth=0.5)
        ax.set_ylabel(name)
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel('Time (s)')
    label = df[df['event'] == event_id]['decision'].values[0]
    fig.suptitle(f"Event: {event_id} — Label: {'TRUE alarm' if label else 'FALSE alarm'} {title_suffix}", fontsize=13)
    plt.tight_layout()
    plt.show()

# Plot one True and one False alarm example
true_event = df[df['decision'] == True].iloc[0]
false_event = df[df['decision'] == False].iloc[0]

plot_example_event(true_event['event'], true_event['record'], DATA_DIR, "(True alarm example)")
plot_example_event(false_event['event'], false_event['record'], DATA_DIR, "(False alarm example)")

The example plots show the raw waveform morphology. Note the differences in ECG patterns between true and false VT alarms — true alarms typically show the wide-complex, rapid rhythm characteristic of ventricular tachycardia.

### Data Cleaning & Preprocessing

Our preprocessing pipeline consists of the following steps, each motivated below:

1. **Waveform loading & channel alignment** — standardise all events to a consistent 4-channel format
2. **Missing signal handling** — zero-fill missing channels and track availability
3. **Signal filtering** — remove noise and artifacts using band-specific filters
4. **Normalization** — z-score standardization

We adapt filtering code from the [VTaC repository](https://github.com/ML-Health/VTaC) (Lehman et al., NeurIPS 2023) and document each step.

#### Step 1: Load waveforms into a consistent format

Each event may have a different set of signals with different naming. We standardise to a fixed 4-channel layout: `[ECG_lead1, ECG_lead2, PLETH, ABP]`. Missing channels are filled with zeros, and we track which channels are present for each event.

In [ ]:
SAMPLING_FREQ = 250
TARGET_SIGNALS = ['ecg1', 'ecg2', 'pleth', 'abp']
# Mapping from common WFDB signal names to our standard channels
ECG_NAMES = {'II', 'I', 'III', 'V', 'AVR', 'AVL', 'AVF', 'MCL', 'ECG'}
PLETH_NAMES = {'PLETH'}
ABP_NAMES = {'ABP'}

def classify_signal(sig_name):
    """Map a WFDB signal name to one of our standard channel types."""
    upper = sig_name.upper().strip()
    if upper in ECG_NAMES:
        return 'ecg'
    elif upper in PLETH_NAMES:
        return 'pleth'
    elif upper in ABP_NAMES:
        return 'abp'
    else:
        return 'unknown'

def load_event_waveform(event_id, record_id, data_dir, target_length=None):
    """
    Load a single event and return a (4, signal_length) array with channels:
    [ECG1, ECG2, PLETH, ABP]. Missing channels are zero-filled.
    
    Also returns a boolean mask of which channels are present.
    """
    wfdb_path = str(data_dir / record_id / event_id)
    rec = wfdb.rdrecord(wfdb_path)
    
    sig_len = rec.sig_len if target_length is None else target_length
    output = np.zeros((4, sig_len), dtype=np.float32)
    available = [False, False, False, False]  # ecg1, ecg2, pleth, abp
    
    ecg_count = 0
    for i, name in enumerate(rec.sig_name):
        signal = rec.p_signal[:, i].astype(np.float32)
        # Truncate or pad to target length
        if len(signal) > sig_len:
            signal = signal[:sig_len]
        
        sig_type = classify_signal(name)
        
        if sig_type == 'ecg' and ecg_count == 0:
            output[0, :len(signal)] = signal
            available[0] = True
            ecg_count += 1
        elif sig_type == 'ecg' and ecg_count == 1:
            output[1, :len(signal)] = signal
            available[1] = True
            ecg_count += 1
        elif sig_type == 'pleth':
            output[2, :len(signal)] = signal
            available[2] = True
        elif sig_type == 'abp':
            output[3, :len(signal)] = signal
            available[3] = True
    
    return output, available

# Test on one event
test_row = df.iloc[0]
waveform, avail = load_event_waveform(test_row['event'], test_row['record'], DATA_DIR)
print(f"Test load — Event: {test_row['event']}")
print(f"  Shape: {waveform.shape}")
print(f"  Channels available: { {TARGET_SIGNALS[i]: avail[i] for i in range(4)} }")

#### Step 2: Load all events & document missing data

In [ ]:
from collections import Counter

def load_all_waveforms(df, data_dir, target_length=None):
    """
    Load all events into arrays. Returns:
      - waveforms: np.array of shape (n_events, 4, signal_length)
      - labels: np.array of shape (n_events,)
      - availability: np.array of shape (n_events, 4) — boolean mask
      - event_ids: list of event ID strings
    """
    waveforms = []
    labels = []
    availability = []
    event_ids = []
    failed = []
    
    for idx, row in df.iterrows():
        try:
            wf, avail = load_event_waveform(row['event'], row['record'], data_dir, target_length)
            waveforms.append(wf)
            labels.append(row['decision'])
            availability.append(avail)
            event_ids.append(row['event'])
        except Exception as e:
            failed.append((row['event'], str(e)))
    
    if failed:
        print(f"WARNING: {len(failed)} events failed to load:")
        for ev, err in failed[:5]:
            print(f"  {ev}: {err}")
    
    return (np.array(waveforms), np.array(labels), np.array(availability), event_ids)

# We determine the expected signal length from the first record
test_row = df.iloc[0]
test_rec = wfdb.rdrecord(str(DATA_DIR / test_row['record'] / test_row['event']))
SIG_LEN = test_rec.sig_len
print(f"Expected signal length: {SIG_LEN} samples ({SIG_LEN/SAMPLING_FREQ:.0f} seconds)")

print("\nLoading all waveforms (this may take a few minutes)...")
waveforms, labels, availability, event_ids = load_all_waveforms(df, DATA_DIR, target_length=SIG_LEN)
print(f"Loaded: {waveforms.shape[0]} events, shape per event: {waveforms.shape[1:]})")

In [ ]:
# Document missing data
print("=== Missing Signal Report ===")
print(f"Total events loaded: {len(waveforms)}")
for i, name in enumerate(TARGET_SIGNALS):
    present = availability[:, i].sum()
    missing = len(waveforms) - present
    print(f"  {name.upper():6s}: {present} present, {missing} missing ({missing/len(waveforms):.1%})")

# How many events have ALL signals vs partial?
n_available = availability.sum(axis=1)
print(f"\n=== Events by number of available channels ===")
for n in sorted(np.unique(n_available)):
    count = (n_available == n).sum()
    print(f"  {int(n)} channels: {count} events ({count/len(waveforms):.1%})")

**Missing data decision:** Events with missing channels are kept rather than deleted. The missing channels remain zero-filled, which acts as an implicit "signal absent" indicator. We track availability so we can later compare model performance with vs. without imputation (assignment requirement 5). Deleting events with any missing signal would reduce our dataset substantially and bias the analysis.

#### Step 3: Signal filtering

We apply signal-specific filters to remove noise and artifacts, adapted from the VTaC repository ([GitHub](https://github.com/ML-Health/VTaC)):

- **ECG (leads 1 & 2):** 1 Hz highpass (remove baseline wander) → 30 Hz lowpass (remove high-freq noise) → 60 Hz notch (remove power line interference)
- **PLETH (PPG):** 60 Hz notch → 0.5–5 Hz bandpass (isolate pulse waveform)
- **ABP:** 60 Hz notch → 16 Hz lowpass (smooth high-freq artifacts)

These filter choices follow standard clinical signal processing practice. The 60 Hz notch filter targets US power line frequency. Filter orders are kept low (order 2) to avoid ringing artifacts.

*Source: filtering adapted from [VTaC preprocessing](https://github.com/ML-Health/VTaC), Lehman et al., NeurIPS 2023.*

In [ ]:
POWERLINE_FREQ = 60

def butter_highpass(cutoff, fs, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='high', analog=False)
    return b, a

def butter_lowpass(cutoff, fs, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low', analog=False)
    return b, a

def notch_filter(freq, Q, fs):
    b, a = iirnotch(freq, Q, fs)
    return b, a

def filter_ecg(signal, fs=SAMPLING_FREQ):
    """Highpass 1 Hz -> Lowpass 30 Hz -> Notch 60 Hz"""
    b, a = butter_highpass(1.0, fs)
    out = filtfilt(b, a, signal)
    b, a = butter_lowpass(30.0, fs)
    out = filtfilt(b, a, out)
    b, a = notch_filter(POWERLINE_FREQ, 30, fs)
    out = filtfilt(b, a, out)
    return out

def filter_ppg(signal, fs=SAMPLING_FREQ):
    """Notch 60 Hz -> Bandpass 0.5–5 Hz"""
    b, a = notch_filter(POWERLINE_FREQ, 30, fs)
    out = filtfilt(b, a, signal)
    b, a = butter(1, [0.5, 5], btype='band', analog=False, fs=fs)
    out = filtfilt(b, a, out)
    return out

def filter_abp(signal, fs=SAMPLING_FREQ):
    """Notch 60 Hz -> Lowpass 16 Hz"""
    b, a = notch_filter(POWERLINE_FREQ, 30, fs)
    out = filtfilt(b, a, signal)
    b, a = butter_lowpass(16.0, fs)
    out = filtfilt(b, a, out)
    return out

# Filter map: channel index -> filter function
FILTER_MAP = {0: filter_ecg, 1: filter_ecg, 2: filter_ppg, 3: filter_abp}

def filter_all(waveforms, availability):
    """Apply channel-specific filters. Only filter channels that are present."""
    filtered = waveforms.copy()
    for ch_idx, filt_fn in FILTER_MAP.items():
        for i in range(len(filtered)):
            if availability[i, ch_idx]:  # only filter if signal is present
                filtered[i, ch_idx] = filt_fn(filtered[i, ch_idx])
    return filtered

print("Applying signal filters...")
waveforms_filtered = filter_all(waveforms, availability)
print("Filtering complete.")

In [ ]:
# Visualise filtering effect on one event
example_idx = 0
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
t = np.arange(waveforms.shape[2]) / SAMPLING_FREQ

for ch, (ax, name) in enumerate(zip(axes, ['ECG Lead 1', 'ECG Lead 2', 'PLETH', 'ABP'])):
    if availability[example_idx, ch]:
        ax.plot(t, waveforms[example_idx, ch], alpha=0.5, linewidth=0.5, label='Raw')
        ax.plot(t, waveforms_filtered[example_idx, ch], linewidth=0.5, label='Filtered')
        ax.legend(loc='upper right', fontsize=8)
    else:
        ax.text(0.5, 0.5, 'Signal not available', transform=ax.transAxes, ha='center', va='center')
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Raw vs Filtered — Event: {event_ids[example_idx]}', fontsize=13)
plt.tight_layout()
plt.show()

#### Step 4: Normalization

We apply **per-sample z-score normalization** — each channel of each event is normalized using its own mean and standard deviation. This is preferred over population-level normalization because signal amplitudes vary significantly between patients and recording setups.

Only channels that are actually present (non-zero) are normalized. Zero-filled missing channels remain at zero.

*Source: standardization approach from [VTaC repository](https://github.com/ML-Health/VTaC).*

In [ ]:
def normalize_per_sample(waveforms, availability):
    """Z-score normalize each channel of each event individually."""
    normed = waveforms.copy()
    for i in range(len(normed)):
        for ch in range(normed.shape[1]):
            if availability[i, ch]:
                mu = normed[i, ch].mean()
                sigma = normed[i, ch].std()
                if sigma > 0:
                    normed[i, ch] = (normed[i, ch] - mu) / sigma
                else:
                    normed[i, ch] = 0.0  # constant signal edge case
    return normed

print("Normalizing...")
waveforms_normed = normalize_per_sample(waveforms_filtered, availability)
print("Normalization complete.")
print(f"Final waveform array shape: {waveforms_normed.shape}")

#### Preprocessing summary

In [ ]:
# Split the processed data back into train/val/test using the original split labels
split_labels = df.loc[df['event'].isin(event_ids), 'split'].values

for split_name in ['train', 'val', 'test']:
    mask = split_labels == split_name
    n = mask.sum()
    n_true = labels[mask].sum()
    print(f"{split_name:5s}: {n} events ({n_true} True, {n - n_true} False, {n_true/n:.1%} True)")

print(f"\nPreprocessing pipeline:")
print(f"  1. Loaded {len(waveforms)} events with 4-channel alignment (zero-fill missing)")
print(f"  2. Applied signal-specific filtering (ECG/PPG/ABP)")
print(f"  3. Per-sample z-score normalization")
print(f"  Final shape: {waveforms_normed.shape} (events × channels × samples)")

### Feature Extraction

*Next: extract features from the preprocessed waveforms for model training. Options include statistical features (mean, std, skewness, kurtosis per channel), frequency-domain features, or using the waveforms directly with a deep learning model.*

## Results and Discussion

This section will contain:
- Results and comparison of model performance
- Summary of best model (name, accuracy, precision, recall, F1-score)
- Impact of imputation strategy on performance
- Feature importance analysis
